# Generate text using the model Qwen3.5-9B

In this notebook, we use the generative large language model
[Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B). This is a hybrid model
which can use "thinking" mode, but it is also possible to omit it.

Qwen3.5-9B has 9 billion parameters, so with 2 bytes per parameter
(`bfloat16`) we expect a memory usage of around 18 GB.

We will use the chat template for Qwen3.5-9B and different prompts. You
can observe the behavior from LLMs that they produce different answers
for the same prompts. You will also see how can you can avoid that
and get reproducible answers.

In [1]:
import torch
torch.cuda.get_device_properties(0) 

_CudaDeviceProperties(name='NVIDIA GeForce RTX 5090', major=12, minor=0, total_memory=32108MB, multi_processor_count=170, uuid=82d3fd6d-eaca-d0e8-c718-917b938713bb, pci_bus_id=38, pci_device_id=0, pci_domain_id=0, L2_cache_size=96MB)

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3.5-9B"

# model and tokenizer must match
model = AutoModelForCausalLM.from_pretrained(model_name, 
                                             dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Qwen3.5 models are *hybrid* models combining normal attention layers with linear attention layers.
This saves a lot of space and computing capacity. To make full use of its acceleration, you have
to also install `flash-linear-attention` and `causal-conv1d`.

In [3]:
# transfer the model to the GPU
model.cuda()

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 4096)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (in_proj_qkv): Linear(in_features=4096, out_features=8192, bias=False)
          (in_proj_z): Linear(in_features=4096, out_features=4096, bias=False)
          (in_proj_b): Linear(in_features=4096, out_features=32, bias=False)
          (in_proj_a): Linear(in_features=4096, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=

Check the memory usage

In [4]:
!nvidia-smi

Tue Sep 15 16:28:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:26:00.0 Off |                  N/A |
|  0%   49C    P1             77W /  575W |   17620MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Prompting the LLM works via the a special array which contains `dict`s. Each `dict`
has two keys, one is for the `role`, the other for the `content`. This structure 
is the same for all LLMs. 

In [5]:
prompt = "Tell me about O'Reilly online learning"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

The template itself however differs considerably. Fortunately,
it is included in the tokenizer.

In [6]:
print(tokenizer.chat_template)

{%- set image_count = namespace(value=0) %}
{%- set video_count = namespace(value=0) %}
{%- macro render_content(content, do_vision_count, is_system_content=false) %}
    {%- if content is string %}
        {{- content }}
    {%- elif content is iterable and content is not mapping %}
        {%- for item in content %}
            {%- if 'image' in item or 'image_url' in item or item.type == 'image' %}
                {%- if is_system_content %}
                    {{- raise_exception('System message cannot contain images.') }}
                {%- endif %}
                {%- if do_vision_count %}
                    {%- set image_count.value = image_count.value + 1 %}
                {%- endif %}
                {%- if add_vision_id %}
                    {{- 'Picture ' ~ image_count.value ~ ': ' }}
                {%- endif %}
                {{- '<|vision_start|><|image_pad|><|vision_end|>' }}
            {%- elif 'video' in item or item.type == 'video' %}
                {%- if is_s

The tokenizer also know how to apply the template and the result is much easier:

In [7]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)
print(text)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Tell me about O'Reilly online learning<|im_end|>
<|im_start|>assistant
<think>

</think>




This can now be used for instructing the LLM to `generate`
which invokes the text completion mechanism.

In [8]:
import time

tokens = tokenizer(text, return_tensors='pt').to(model.device)
start = time.time()
output = model.generate(**tokens,
                        temperature=0.7, 
                        do_sample=True, top_p=0.95, top_k=40, 
                        max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
print(tokenizer.decode(output[0]))
print(f"{used} seconds, {tps} tokens/s")

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Tell me about O'Reilly online learning<|im_end|>
<|im_start|>assistant
<think>

</think>

**O'Reilly Online Learning** is a premier digital learning platform dedicated to technical skills, primarily catering to software developers, IT professionals, data scientists, and digital marketers. Owned by the same company behind the famous "O'Reilly Media" books (renowned for their animal-covered book covers), the platform bridges the gap between traditional publishing and modern, interactive learning.

Here is a breakdown of what makes O'Reilly Online Learning stand out:

### 1. Core Content & Formats
Unlike generic video courses found on YouTube or Udemy, O'Reilly focuses heavily on **text-based learning enriched with interactive elements**.
*   **Interactive Ebooks**: Their flagship content is the "interactive ebook." These are not just PDFs; they allow you to run code snippets directly within the browser, search the

This does not look *nice*. The prompt is repeated and the formatting looks weird.
We can skip the prompt and format the text as Markdown to make it easier to read:

In [9]:
from IPython.display import Markdown, display
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))

**O'Reilly Online Learning** is a premier digital learning platform dedicated to technical skills, primarily catering to software developers, IT professionals, data scientists, and digital marketers. Owned by the same company behind the famous "O'Reilly Media" books (renowned for their animal-covered book covers), the platform bridges the gap between traditional publishing and modern, interactive learning.

Here is a breakdown of what makes O'Reilly Online Learning stand out:

### 1. Core Content & Formats
Unlike generic video courses found on YouTube or Udemy, O'Reilly focuses heavily on **text-based learning enriched with interactive elements**.
*   **Interactive Ebooks**: Their flagship content is the "interactive ebook." These are not just PDFs; they allow you to run code snippets directly within the browser, search the text, and bookmark sections easily.
*   **Video Courses**: They offer structured video courses led by industry experts, often focusing on specific tools, frameworks, or career paths (e.g., "Python for Data Science," "AWS Solutions Architect," or "React Native").
*   **Live Events & Webinars**: They host live technical events, conferences (like O'Reilly Open Source Conference), and on-demand webinars featuring top voices in technology.

### 2. Key Subject Areas
The curriculum is vast but heavily weighted toward technology:
*   **Software Development**: Languages like Python, Java, JavaScript, Go, and Rust.
*   **Data & AI**: Data science, machine learning, big data, and cloud computing.
*   **IT & DevOps**: Cloud platforms (AWS, Azure, GCP), Kubernetes, Docker, and site reliability engineering.
*   **Business & Marketing**: Digital marketing, product management, and business analytics.

### 3. Learning Pathways
One of the platform's most popular features is its **"Learning Pathways."** Instead of just buying individual courses, users can select a career goal (e.g., "Become a DevOps Engineer" or "Master Machine Learning"). The platform then curates a sequence of courses, books, and resources tailored to take you from beginner to advanced proficiency in that specific field.

### 4. Target Audience
*   **Individual Professionals**: For self-paced upskilling or reskilling.
*   **Enterprises**: O'Reilly offers enterprise licenses that allow companies to provide unlimited access to their entire library for their employees, often including integration with internal Learning Management Systems (LMS).
*  

Due to the positive temperature, we get different responses (sampling):

In [ ]:
start = time.time()
output = model.generate(**tokens,
                        temperature=0.7, 
                        do_sample=True, top_p=0.95, top_k=40, 
                        max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

If we set `do_sample=False`, we get reproducible results:

In [ ]:
start = time.time()

output = model.generate(**tokens,
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

In [ ]:
start = time.time()

output = model.generate(**tokens,
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

Take a look at what happens when we ask for knowledge which is not in the training data:

In [ ]:
prompt = "Explain the QXGL training method for LLMs!"
messages = [
    {"role": "system", "content": "You are a helpful assistant. Only answer if you are absolutely sure. Otherwise tell me that you don't know the answer"},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)
tokens = tokenizer(text, return_tensors='pt').to(model.device)

start = time.time()
output = model.generate(**tokens,
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

Slight change: take a look at the system prompt!

In [ ]:
prompt = "Write an abstract about 'Frontiers in QXGL training LLMs'!"
messages = [
    {"role": "system", "content": "You are a creative researcher."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)
tokens = tokenizer(text, return_tensors='pt').to(model.device)

start = time.time()
output = model.generate(inputs=tokens.input_ids.cuda(), attention_mask=tokens.attention_mask.cuda(),
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")

In [ ]:
prompt = "Write an abstract about 'Frontiers in QXGL training LLMs'!"
messages = [
    {"role": "system", "content": "Talk like you are a drunk pirate."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)
tokens = tokenizer(text, return_tensors='pt').to(model.device)

start = time.time()
output = model.generate(inputs=tokens.input_ids.cuda(), attention_mask=tokens.attention_mask.cuda(),
                        do_sample=False, max_new_tokens=512)
used = time.time() - start
tps = len(output[0]) / used
display(Markdown(tokenizer.decode(output[0][len(tokens.input_ids[0]):])))
print(f"{used} seconds, {tps} tokens/s")